# 12_GFS_Forecast_Extraction_v4
Uses Open-Meteo GFS JSON API (no xarray), geocodes a city, summarizes 72-hour forecast into features compatible with the weather branch.

In [ ]:
!pip -q install geopy requests pandas numpy

import requests
import pandas as pd
import numpy as np
from geopy.geocoders import Nominatim
from google.colab import files

city=input("Enter City Name: ").strip()

geo=Nominatim(user_agent="heatwave-flood-prediction")
loc=geo.geocode(city)

if loc is None:
    raise ValueError("City not found.")

lat=loc.latitude
lon=loc.longitude

print(f"City: {city}")
print(f"Latitude: {lat}")
print(f"Longitude: {lon}")

url=(
    f"https://api.open-meteo.com/v1/gfs?"
    f"latitude={lat}"
    f"&longitude={lon}"
    f"&hourly=temperature_2m,relative_humidity_2m,"
    f"dew_point_2m,pressure_msl,wind_speed_10m,"
    f"wind_direction_10m,precipitation"
    f"&forecast_days=3"
    f"&timezone=auto"
)

print("Requesting forecast...")
resp=requests.get(url,timeout=60)
resp.raise_for_status()

data=resp.json()
hourly=data["hourly"]

df=pd.DataFrame({
    "time":hourly["time"],
    "temperature_2m":hourly["temperature_2m"],
    "relative_humidity_2m":hourly["relative_humidity_2m"],
    "dew_point_2m":hourly["dew_point_2m"],
    "pressure_msl":hourly["pressure_msl"],
    "wind_speed_10m":hourly["wind_speed_10m"],
    "wind_direction_10m":hourly["wind_direction_10m"],
    "precipitation":hourly["precipitation"]
})

df["u_wind"]=-df["wind_speed_10m"]*np.sin(np.radians(df["wind_direction_10m"]))
df["v_wind"]=-df["wind_speed_10m"]*np.cos(np.radians(df["wind_direction_10m"]))

summary=pd.DataFrame([{
    "city":city,
    "latitude":lat,
    "longitude":lon,
    "temperature_mean":df["temperature_2m"].mean(),
    "humidity_mean":df["relative_humidity_2m"].mean(),
    "dew_point_mean":df["dew_point_2m"].mean(),
    "pressure_mean":df["pressure_msl"].mean(),
    "u_wind_mean":df["u_wind"].mean(),
    "v_wind_mean":df["v_wind"].mean(),
    "precipitation_sum":df["precipitation"].sum()
}])

df.to_csv("gfs_hourly_forecast.csv",index=False)
summary.to_csv("gfs_features.csv",index=False)

print("\nHourly forecast:")
print(df.head())
print("\nSummary features:")
print(summary)

files.download("gfs_hourly_forecast.csv")
files.download("gfs_features.csv")
